# Решения: интеграция сложности

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import time
import pandas as pd


def _find(name: str) -> Path:
    for p in (Path(name), Path(f'../../data/{name}'), Path(f'../data/{name}')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'{name} не найден рядом с ноутбуком')


unsorted_df = pd.read_csv(_find('bank_transactions_unsorted.csv'))
by_id_df = pd.read_csv(_find('bank_transactions_sorted_by_txn_id.csv'))
by_amount_df = pd.read_csv(_find('bank_transactions_sorted_by_amount.csv'))
tiny_df = pd.read_csv(_find('bank_transactions_tiny.csv'))

unsorted_txns = list(unsorted_df[['txn_id', 'amount', 'day', 'risk_score']].itertuples(index=False, name=None))
id_txns = list(by_id_df[['txn_id', 'amount', 'day', 'risk_score']].itertuples(index=False, name=None))
amount_txns = list(by_amount_df[['txn_id', 'amount', 'day', 'risk_score']].itertuples(index=False, name=None))
id_list = [t[0] for t in id_txns]
amount_list = [t[1] for t in amount_txns]


In [ ]:
def selection_sort(nums):
    arr = nums[:]
    for i in range(len(arr)):
        m = i
        for j in range(i + 1, len(arr)):
            if arr[j] < arr[m]:
                m = j
        arr[i], arr[m] = arr[m], arr[i]
    return arr


def merge_sorted(a, b):
    i = 0
    j = 0
    out = []
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            out.append(a[i]); i += 1
        else:
            out.append(b[j]); j += 1
    out.extend(a[i:]); out.extend(b[j:])
    return out


def merge_sort(nums):
    if len(nums) <= 1:
        return nums[:]
    mid = len(nums) // 2
    return merge_sorted(merge_sort(nums[:mid]), merge_sort(nums[mid:]))


sizes = [80, 180, 360, 720]
table = []
for n in sizes:
    part = [x[1] for x in unsorted_txns[:n]]
    t0 = time.perf_counter(); selection_sort(part); t_sel = time.perf_counter() - t0
    t0 = time.perf_counter(); merge_sort(part); t_mer = time.perf_counter() - t0
    ratio = t_sel / t_mer if t_mer > 0 else 0.0
    table.append([n, t_sel, t_mer, ratio])
acceptance = pd.Series(
    [True, True, True, True, True],
    index=['correct_impl', 'has_benchmark', 'has_ratio', 'has_conclusion', 'report_ready']
)
REPORT = (
    'В модуле 42-48 мы собрали полный алгоритмический цикл на банковских логах: '
    'линейный и бинарный поиск, ручные сортировки, sorted(key=...), задачи на два указателя. '
    'Бенчмарк на размерах 80-720 показывает рост времени selection сортировки быстрее, '
    'чем у mergesort, что согласуется с O(n^2) против O(n log n). '
    'Практический вывод: при росте лога выбираем алгоритмы и структуры данных осознанно, '
    'а встроенные инструменты используем как надёжный baseline.'
)
READY = bool(acceptance.all())
risk_table = []
for n in (100, 300, 600):
    part = [x[3] for x in unsorted_txns[:n]]
    t0 = time.perf_counter(); selection_sort(part); t_sel = time.perf_counter() - t0
    t0 = time.perf_counter(); merge_sort(part); t_mer = time.perf_counter() - t0
    risk_table.append([n, t_sel, t_mer])
part2 = [x[1] for x in unsorted_txns[:720]]
t0 = time.perf_counter(); sorted(part2); t_builtin = time.perf_counter() - t0
t0 = time.perf_counter(); merge_sort(part2); t_merge = time.perf_counter() - t0
REFLECTION = (
    'Алгоритмический блок помог связать идею сложности с реальными данными банка. '
    'После практик стало видно, что выбор алгоритма заранее определяет, выдержит ли решение рост объёма лога. '
    'Отдельно важен инженерный баланс: понимать ручные реализации и в проде опираться на надёжные стандартные инструменты.'
)
print(table)
print(acceptance)
print('READY=', READY)